In [2]:
%pip install dotenv


  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install langchain langchain-openai


  Using cached langchain-1.1.0-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_openai-1.1.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached openai-2.8.1-py3-none-any.whl.metadata (29 kB)
  Using cached tiktoken-0.12.0-cp312-cp312-win_amd64.whl.metadata (6.9 kB)
  Using cached langgraph_checkpoint-3.0.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached langgraph_prebuilt-1.0.5-py3-none-any.whl.metadata (5.2 kB)
  Using cached xxhash-3.6.0-cp312-cp312-win_amd64.whl.metadata (13 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.12.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached regex-2025.11.3-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached ormsgpack-1.12.0-cp312-cp312-win_amd64.whl.metadata (1.2 kB)
Using cached langchain-1.1.0-py3-none-any.whl (101 kB)
Using cached langchain_openai-1.1.0-py3-none-any.whl (84 kB)
   ------------------------------------


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Literal
from typing_extensions import Annotated

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from IPython.display import Image, display

# Load env
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    api_key=api_key
)

In [7]:
%pip install deepagents tavily-python

  Using cached wcmatch-10.1-py3-none-any.whl.metadata (5.1 kB)
  Using cached bracex-2.6-py3-none-any.whl.metadata (3.6 kB)
  Using cached docstring_parser-0.17.0-py3-none-any.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/52.0 kB ? eta -:--:--
   ---------------------------------------- 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/49.5 kB ? eta -:--:--
   --------------------------------- ------ 41.0/49.5 kB 1.9 MB/s eta 0:00:01
   --------------------------------- ------ 41.0/49.5 kB 1.9 MB/s eta 0:00:01
   --------------------------------- ------ 41.0/49.5 kB 1.9 MB/s eta 0:00:01
   --------------------------------- ------ 41.0/49.5 kB 1.9 MB/s eta 0:00:01
   --------------------------------- ------ 41.0/49.5 kB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 49.5/49.5 kB 157.1 kB/s eta 0:00:00
Using cached wcmatch-10.1-py3-none-any.whl (39 kB)
   ---------------------------------------- 0.0/388.2 kB ? eta 


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
import os
from typing import Literal
from tavily import TavilyClient
from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI


# Load env
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# Initialize OpenAI model
llm = ChatOpenAI(
    model="gpt-4o-mini",  # or any supported model
    api_key=os.environ["OPENAI_API_KEY"],
)

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Search the internet using Tavily"""
    return tavily_client.search(
        query=query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )



In [35]:
# System prompt to steer the agent to be an expert researcher
research_instructions = """You are an expert researcher. Your job is to conduct thorough research and then write a polished report.

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

agent = create_deep_agent(
    tools=[internet_search],
    system_prompt=research_instructions,
)

In [37]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is langgraph?"}]})

# Print the agent's response
print(result["messages"][-1].content)

TypeError: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

In [45]:
import os
import re
from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI

# 1) Define the tool
def convert_cpp_to_documentation(code: str) -> str:
    """
    Convert C++ source code into legacy engineering documentation format.
    """
    match = re.search(r"(?:int|float|double|void|char|bool|long|short)\s+(\w+)\s*\((.*?)\)", code, re.S)
    function_name = match.group(1) if match else "UNKNOWN"
    params_raw = match.group(2) if match else ""

    params = []
    if params_raw.strip():
        for arg in params_raw.split(","):
            parts = arg.strip().split()
            if len(parts) == 2:
                p_type, p_name = parts
                params.append((p_name, p_type))
            else:
                params.append((arg.strip(), "UNKNOWN"))

    param_doc = "\n".join([f"    {name:<10} : {ptype}" for name, ptype in params]) if params else "    None"

    legacy_doc = f"""
--------------------------------------------------------------------
MODULE DOCUMENTATION
--------------------------------------------------------------------
FUNCTION NAME   : {function_name}
PURPOSE         : Auto-generated documentation for legacy migration.
FILE LOCATION   : Not specified
--------------------------------------------------------------------
INPUT PARAMETERS
{param_doc}
--------------------------------------------------------------------
OUTPUT
    UNKNOWN (Auto-detected based on code)
--------------------------------------------------------------------
DETAILED DESCRIPTION
    This function is automatically documented from source code.
    Please update description content manually if required.
--------------------------------------------------------------------
ALGORITHM (INFERRED)
    The function executes the logic as written in the C++ definition.
--------------------------------------------------------------------
SOURCE CODE
--------------------------------------------------------------------
{code.strip()}
--------------------------------------------------------------------
ERROR / EXCEPTION BEHAVIOR
    No explicit exception or error handling detected.
--------------------------------------------------------------------
DEPENDENCIES
    None inferred from code block.
--------------------------------------------------------------------
REVISION HISTORY
    DATE        VERSION     AUTHOR            DESCRIPTION
    AUTO        1.0         AUTO-GENERATED    Initial migration documentation
--------------------------------------------------------------------
"""
    return legacy_doc.strip()


# 2) Build agent with the tool
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"]
)

system_prompt = """
You are a legacy engineering documentation generator.
When the user gives you C or C++ code, use the tool `convert_cpp_to_documentation` to generate documentation.
Do not produce other text yourself if using code input.
"""

agent = create_deep_agent(
    model=llm,
    tools=[convert_cpp_to_documentation],
    system_prompt=system_prompt
)

# 3) Use the agent
cpp_code = """int add(int a,int b){ return a + b; }"""

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": f"Please generate legacy documentation for this C++ function:\n{cpp_code}"
        }
    ]
})

print(response["messages"][-1].content)


```plaintext
--------------------------------------------------------------------
MODULE DOCUMENTATION
--------------------------------------------------------------------
FUNCTION NAME   : add
PURPOSE         : Auto-generated documentation for legacy migration.
FILE LOCATION   : Not specified
--------------------------------------------------------------------
INPUT PARAMETERS
    a          : int
    b          : int
--------------------------------------------------------------------
OUTPUT
    UNKNOWN (Auto-detected based on code)
--------------------------------------------------------------------
DETAILED DESCRIPTION
    This function is automatically documented from source code.
    Please update description content manually if required.
--------------------------------------------------------------------
ALGORITHM (INFERRED)
    The function executes the logic as written in the C++ definition.
--------------------------------------------------------------------
SOURCE CODE
----

In [46]:
%pip install pythonnet


  Using cached pycparser-2.23-py3-none-any.whl.metadata (993 bytes)
   ---------------------------------------- 0.0/297.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/297.5 kB ? eta -:--:--
   - -------------------------------------- 10.2/297.5 kB ? eta -:--:--
   - -------------------------------------- 10.2/297.5 kB ? eta -:--:--
   ----- --------------------------------- 41.0/297.5 kB 326.8 kB/s eta 0:00:01
   -------------------------- ------------- 194.6/297.5 kB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 297.5/297.5 kB 1.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/56.4 kB ? eta -:--:--
   ---------------------------------------- 56.4/56.4 kB 2.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/183.6 kB ? eta -:--:--
   ----------------- ---------------------- 81.9/183.6 kB 4.5 MB/s eta 0:00:01
   ---------------------------------------- 183.6/183.6 kB 2.8 MB/s eta 0:00:00
Using cached pycparser-2.23-


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [48]:
import clr
import os

# Use current working directory
script_dir = os.getcwd()

# Add Roslyn DLLs
clr.AddReference(os.path.join(script_dir, "Microsoft.CodeAnalysis.CSharp.dll"))
clr.AddReference(os.path.join(script_dir, "Microsoft.CodeAnalysis.dll"))

from Microsoft.CodeAnalysis.CSharp import CSharpSyntaxTree
from Microsoft.CodeAnalysis import SyntaxKind

# Sample C# code
csharp_code = """
/// <summary>
/// Adds two integers
/// </summary>
int Add(int a, int b) {
    return a + b;
}
"""

# Parse C# code
tree = CSharpSyntaxTree.ParseText(csharp_code)
root = tree.GetRoot()

# Traverse methods
for node in root.DescendantNodes():
    if node.IsKind(SyntaxKind.MethodDeclaration):
        print("Method name:", node.Identifier.ValueText)
        print("Return type:", node.ReturnType.ToString())
        print("Parameters:")
        for param in node.ParameterList.Parameters:
            print(f"  {param.Identifier.ValueText} : {param.Type.ToString()}")
        # Print XML documentation comments
        trivia = node.GetLeadingTrivia()
        for t in trivia:
            if t.IsKind(SyntaxKind.SingleLineDocumentationCommentTrivia):
                print("Doc comment:", t.ToString())


FileNotFoundException: Unable to find assembly 'c:\Users\sam\Documents\projects\rag-project\langgraph-rag\src\poc1\backend\Microsoft.CodeAnalysis.CSharp.dll'.
   at Python.Runtime.CLRModule.AddReference(String name)

In [ ]:
from pycparser import c_parser, c_ast

code = """
int add(int a, int b) {
    return a + b;
}
"""

parser = c_parser.CParser()
ast = parser.parse(code)

class FuncVisitor(c_ast.NodeVisitor):
    def visit_FuncDef(self, node):
        # Function name
        func_name = node.decl.name
        print("Function name:", func_name)

        # Return type
        # node.decl.type -> FuncDecl
        # node.decl.type.type -> TypeDecl
        # node.decl.type.type.type -> IdentifierType
        return_type = 'UNKNOWN'
        if hasattr(node.decl.type, 'type') and hasattr(node.decl.type.type, 'names'):
            return_type = ' '.join(node.decl.type.type.names)
        print("Return type:", return_type)

        # Parameters
        params = []
        if node.decl.type.args:
            for p in node.decl.type.args.params:
                param_type = ' '.join(p.type.type.names)
                param_name = p.name
                params.append((param_name, param_type))
        print("Parameters:", params)

v = FuncVisitor()
v.visit(ast)


Function name: add
Return type: UNKNOWN
Parameters: [('a', 'int'), ('b', 'int')]


: 